In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!ls scripts

/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 202 (delta 66), reused 178 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 142.70 KiB | 4.92 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/RiskAware-Complaints-Engine
error_analysis_category.py  prepare_lstm_data.py    tune_category_optuna.py
evaluate_category.py	    train_category.py	    tune_category.py
prepare_data.py		    train_lstm_category.py


In [14]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 859 bytes | 429.00 KiB/s, done.
From https://github.com/Nusultan11/RiskAware-Complaints-Engine
   a390c2e..4df2828  master     -> origin/master
Updating a390c2e..4df2828
error: Your local changes to the following files would be overwritten by merge:
	configs/category.yaml
Please commit your changes or stash them before you merge.
Aborting


In [2]:
import os, glob, shutil,pathlib,yaml, torch, zipfile
from google.colab import files,drive
from pathlib import Path

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ROOT = "/content/RiskAware-Complaints-Engine"
os.makedirs(f"{ROOT}/data/raw", exist_ok=True)
matches = glob.glob("/content/drive/MyDrive/**/cfpb_complaints.csv", recursive=True)
print("matches:", matches)

if not matches:
    raise FileNotFoundError("cfpb_complaints.csv не найден в MyDrive")

SRC = matches[0]
DST = f"{ROOT}/data/raw/cfpb_complaints.csv"
shutil.copy(SRC, DST)

print("copied:", SRC, "->", DST)
print("exists:", os.path.exists(DST))

matches: ['/content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv']
copied: /content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv -> /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv
exists: True


In [6]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True Tesla T4


In [7]:
!PYTHONPATH=src python scripts/prepare_data.py
!PYTHONPATH=src python scripts/prepare_lstm_data.py
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 12
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

LSTM preprocessing completed.
Saved tensors to: artifacts/lstm_preprocessing
device=cuda use_class_weights=True learning_rate=0.0003
epoch=1 train_loss=4.5857 val_macro_f1=0.0152 val_accuracy=0.0738
epoch=2 train_loss=4.4991 val_macro_f1=0.0205 val_accuracy=0.1122
epoch=3 train_loss=4.0370 val_macro_f1=0.0424 val_accuracy=0.1652
epoch=4 train_loss=3.6565 val_macro_f1=0.0508 val_accuracy=0.1583
epoch=5 train_loss=3.3505 val_macro_f1=0.0610 val_accuracy=0.1635
epoch=6 train_loss=3.0784 val_macro_f1=0.0686 val_accuracy=0.1764
epoch=7 train_loss=2.8100 val_macro_f1=0.0870 val_accuracy=0.1938
epoch=8 train_loss=2.5806 val_macro_f1=0.0809 val_accuracy=0.2052
epoch=9 train_loss=2.3403 val_macro_f1=0.0815 val_accuracy=0.1980
epoch=10 train_loss=2.1670 val_macro_f1=0.0947 val_accuracy=0.2059
epoch=11 train_loss=1.9810 val_macro_f1=0.1012 val_accuracy=0.2258
epoch=12 train_loss=1.8420 val_macro_f1=0.1098 val_accuracy=0.2418
LSTM category training completed.
Best epoch: 12
Val Macro-F1: 0.1098
Te

In [9]:
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 20
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

device=cuda architecture=bilstm_only use_class_weights=False learning_rate=0.001
epoch=1 train_loss=3.1959 val_macro_f1=0.0228 val_accuracy=0.3016
epoch=2 train_loss=2.4470 val_macro_f1=0.0449 val_accuracy=0.3757
epoch=3 train_loss=2.1203 val_macro_f1=0.0709 val_accuracy=0.4188
epoch=4 train_loss=1.8972 val_macro_f1=0.1068 val_accuracy=0.4501
epoch=5 train_loss=1.7358 val_macro_f1=0.1273 val_accuracy=0.4690
epoch=6 train_loss=1.5943 val_macro_f1=0.1427 val_accuracy=0.4754
epoch=7 train_loss=1.4744 val_macro_f1=0.1485 val_accuracy=0.4861
epoch=8 train_loss=1.3623 val_macro_f1=0.1577 val_accuracy=0.4866
epoch=9 train_loss=1.2559 val_macro_f1=0.1637 val_accuracy=0.4831
epoch=10 train_loss=1.1487 val_macro_f1=0.1719 val_accuracy=0.4897
epoch=11 train_loss=1.0497 val_macro_f1=0.1756 val_accuracy=0.4843
epoch=12 train_loss=0.9461 val_macro_f1=0.1767 val_accuracy=0.4913
epoch=13 train_loss=0.8527 val_macro_f1=0.1756 val_accuracy=0.4899
epoch=14 train_loss=0.7639 val_macro_f1=0.1813 val_accura

In [10]:
bundle = Path("lstm_v2_bundle.zip")
paths = [
    Path("artifacts/category_lstm"),
    Path("artifacts/lstm_preprocessing/metadata.json"),
    Path("artifacts/lstm_preprocessing/vocab.json"),
    Path("reports/metrics/category_lstm_metrics_val.json"),
    Path("reports/metrics/category_lstm_metrics_test.json"),
    Path("configs/category.yaml"),
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.is_file():
            z.write(p, arcname=str(p))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f))

print("Created:", bundle.resolve())

Created: /content/RiskAware-Complaints-Engine/lstm_v2_bundle.zip


In [11]:
files.download("/content/RiskAware-Complaints-Engine/lstm_v2_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
cfg_path = pathlib.Path("/content/RiskAware-Complaints-Engine/configs/category.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["stacks"]["bilstm"]["use_class_weights"] = True
cfg["stacks"]["bilstm"]["class_weight_max"] = 3.0
cfg["stacks"]["bilstm"]["learning_rate"] = 0.001
cfg["stacks"]["bilstm"]["epochs"] = 20
cfg["stacks"]["bilstm"]["early_stopping_patience"] = 3

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("updated A")

updated A


In [13]:
!PYTHONPATH=src python scripts/prepare_lstm_data.py
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 20
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

LSTM preprocessing completed.
Saved tensors to: artifacts/lstm_preprocessing
device=cuda architecture=bilstm_only use_class_weights=True learning_rate=0.001
epoch=1 train_loss=4.5717 val_macro_f1=0.0288 val_accuracy=0.1186
epoch=2 train_loss=4.0434 val_macro_f1=0.0536 val_accuracy=0.1702
epoch=3 train_loss=3.3468 val_macro_f1=0.0707 val_accuracy=0.1916
epoch=4 train_loss=2.7967 val_macro_f1=0.0742 val_accuracy=0.1646
epoch=5 train_loss=2.3608 val_macro_f1=0.0924 val_accuracy=0.1979
epoch=6 train_loss=2.0227 val_macro_f1=0.1040 val_accuracy=0.2258
epoch=7 train_loss=1.7382 val_macro_f1=0.1226 val_accuracy=0.2327
epoch=8 train_loss=1.4962 val_macro_f1=0.1309 val_accuracy=0.2605
epoch=9 train_loss=1.3032 val_macro_f1=0.1419 val_accuracy=0.2473
epoch=10 train_loss=1.1617 val_macro_f1=0.1516 val_accuracy=0.2755
epoch=11 train_loss=1.0289 val_macro_f1=0.1515 val_accuracy=0.2663
epoch=12 train_loss=0.9156 val_macro_f1=0.1654 val_accuracy=0.3174
epoch=13 train_loss=0.8255 val_macro_f1=0.1570 v

In [15]:
!git stash
!git pull
!git stash pop

Saved working directory and index state WIP on master: a390c2e refactor(lstm): support bilstm-only architecture
Updating a390c2e..4df2828
Fast-forward
 configs/category.yaml          |  3 +++
 scripts/train_lstm_category.py | 18 +++++++++++++++++-
 2 files changed, 20 insertions(+), 1 deletion(-)
Auto-merging configs/category.yaml
CONFLICT (content): Merge conflict in configs/category.yaml
On branch master
Your branch is up to date with 'origin/master'.

Unmerged paths:
  (use "git restore --staged <file>..." to unstage)
  (use "git add <file>..." to mark resolution)
	both modified:   configs/category.yaml

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	lstm_v2_bundle.zip

no changes added to commit (use "git add" and/or "git commit -a")
The stash entry is kept in case you need it again.


In [17]:
!git config user.name "Nursultan11"
!git config user.email "ortajnursultan491@gmail.com"
!git commit -m "chore: resolve category config merge conflict"

[master 0ff65a6] chore: resolve category config merge conflict
 1 file changed, 3 insertions(+), 6 deletions(-)


In [18]:
!PYTHONPATH=src python scripts/prepare_lstm_data.py
!PYTHONPATH=src python scripts/train_lstm_category.py --epochs 20
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

LSTM preprocessing completed.
Saved tensors to: artifacts/lstm_preprocessing
device=cuda architecture=bilstm_only use_class_weights=True learning_rate=0.001
epoch=1 train_loss=4.5717 val_macro_f1=0.0288 val_accuracy=0.1186 lr=0.001000
epoch=2 train_loss=4.0434 val_macro_f1=0.0536 val_accuracy=0.1702 lr=0.001000
epoch=3 train_loss=3.3468 val_macro_f1=0.0707 val_accuracy=0.1916 lr=0.001000
epoch=4 train_loss=2.7967 val_macro_f1=0.0742 val_accuracy=0.1646 lr=0.001000
epoch=5 train_loss=2.3608 val_macro_f1=0.0924 val_accuracy=0.1979 lr=0.001000
epoch=6 train_loss=2.0227 val_macro_f1=0.1040 val_accuracy=0.2258 lr=0.001000
epoch=7 train_loss=1.7382 val_macro_f1=0.1226 val_accuracy=0.2327 lr=0.001000
epoch=8 train_loss=1.4962 val_macro_f1=0.1309 val_accuracy=0.2605 lr=0.001000
epoch=9 train_loss=1.3032 val_macro_f1=0.1419 val_accuracy=0.2473 lr=0.001000
epoch=10 train_loss=1.1617 val_macro_f1=0.1516 val_accuracy=0.2755 lr=0.001000
epoch=11 train_loss=1.0289 val_macro_f1=0.1515 val_accuracy=0.

In [19]:
bundle = Path("lstm_v3_scheduler_bundle.zip")
paths = [
    Path("artifacts/category_lstm"),
    Path("artifacts/lstm_preprocessing/metadata.json"),
    Path("artifacts/lstm_preprocessing/vocab.json"),
    Path("reports/metrics/category_lstm_metrics_val.json"),
    Path("reports/metrics/category_lstm_metrics_test.json"),
    Path("configs/category.yaml"),
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.is_file():
            z.write(p, arcname=str(p))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f))

print("Created:", bundle.resolve())

Created: /content/RiskAware-Complaints-Engine/lstm_v3_scheduler_bundle.zip


In [20]:
files.download("/content/RiskAware-Complaints-Engine/lstm_v3_scheduler_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>